In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install --no-deps trl peft accelerate bitsandbytes -q

In [ ]:
import os, shutil
from unsloth import FastLanguageModel

# Check free disk space first
import shutil as sh
total, used, free = sh.disk_usage("/kaggle/working")
print(f"Free: {free/1e9:.1f} GB — need ~15 GB for f16 intermediate + final Q4_K_M")

In [ ]:
# ── Convert DG adapter ────────────────────────────────────────────
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="/kaggle/input/datasets/seifmahdy1/mcq-dg-adapter/results (2)/mcq_dg/adapter",
    max_seq_length=512,
    load_in_4bit=True,
)
model.save_pretrained_merged(
    "/kaggle/working/mcq_dg_merged",
    tokenizer,
    save_method="merged_16bit",
)
print("✅ Merged weights saved")
# Free GPU memory
import torch
del model
torch.cuda.empty_cache()

In [ ]:
import os, shutil, subprocess

!df -h /kaggle/working

# ── Nuke any previous broken llama.cpp clone ─────────────────────
if os.path.exists("/root/.unsloth/llama.cpp"):
    shutil.rmtree("/root/.unsloth/llama.cpp")
    print("🗑️  Deleted old broken llama.cpp clone")

# ── Fresh clone ──────────────────────────────────────────────────
subprocess.run([
    "git", "clone", "--depth=1",
    "https://github.com/ggml-org/llama.cpp",
    "/root/.unsloth/llama.cpp"
], check=True)
print("✅ Cloned llama.cpp")

# ── Build with CUDA OFF — we only need the CPU quantizer ────────
subprocess.run([
    "cmake",
    "-B", "/root/.unsloth/llama.cpp/build",
    "-S", "/root/.unsloth/llama.cpp",
    "-DCMAKE_BUILD_TYPE=Release",
    "-DGGML_CUDA=OFF",
    "-DGGML_METAL=OFF",
    "-DGGML_VULKAN=OFF",
    "-DGGML_SYCL=OFF",
    "-DGGML_CCACHE=OFF",
    "-DLLAMA_BUILD_TESTS=OFF",
    "-DLLAMA_BUILD_EXAMPLES=OFF",
    "-DLLAMA_BUILD_SERVER=OFF",
], check=True)
print("✅ CMake configured (CPU only)")

subprocess.run([
    "cmake", "--build", "/root/.unsloth/llama.cpp/build",
    "--config", "Release", "-j4",
    "-t", "llama-quantize",
], check=True)
print("✅ llama-quantize built")

# ── Install the gguf Python package for the converter script ─────
subprocess.run(["pip", "install", "-q", "gguf"], check=True)

# ── Convert merged HF weights → f16 GGUF ────────────────────────
subprocess.run([
    "python3", "/root/.unsloth/llama.cpp/convert_hf_to_gguf.py",
    "/kaggle/working/mcq_dg_merged",
    "--outfile", "/kaggle/working/mcq_dg.f16.gguf",
    "--outtype", "f16",
], check=True)
print("✅ f16 GGUF conversion done")

# ── Free disk: delete merged weights (~6 GB) before quantizing ───
shutil.rmtree("/kaggle/working/mcq_dg_merged")
print("🗑️  Deleted merged weights")
!df -h /kaggle/working

# ── Quantize f16 → Q4_K_M ───────────────────────────────────────
subprocess.run([
    "/root/.unsloth/llama.cpp/build/bin/llama-quantize",
    "/kaggle/working/mcq_dg.f16.gguf",
    "/kaggle/working/mcq_dg.Q4_K_M.gguf",
    "Q4_K_M",
], check=True)
print("✅ Q4_K_M quantization done")

# ── Clean up f16 intermediate ────────────────────────────────────
os.remove("/kaggle/working/mcq_dg.f16.gguf")

# ── Write Ollama Modelfile ───────────────────────────────────────
with open("/kaggle/working/mcq_dg_Modelfile", "w") as f:
    f.write("FROM ./mcq_dg.Q4_K_M.gguf\n")
    f.write("PARAMETER temperature 0.0\n")
    f.write("PARAMETER num_ctx 512\n")
    f.write("PARAMETER num_predict 80\n")

print("\n✅ ALL DONE — DG GGUF ready for download")
!ls -lh /kaggle/working/mcq_dg*